# ACDC: ejection fraction and segmentation metrics

Run from the `notebooks/` folder after `pip install -e .`. Paths are relative to it: datasets live in `../data/`, results go to `../output/`.

Requires the [ACDC dataset](https://www.creatis.insa-lyon.fr/Challenge/acdc/) extracted to `../data/acdc/training/patientXXX/` and `pip install -e ".[notebooks]"` (for nibabel).

In [ ]:
import csv
import glob
import os

import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

from implicit_heart import ImplicitFunction
from implicit_heart import metrics
from implicit_heart.acdc import load_frames

## Fit each patient and estimate ejection fraction

Labels: 1 = RV, 2 = myocardium, 3 = LV. Only the LV is kept.

In [ ]:
os.makedirs("../output/acdc", exist_ok=True)
output_csv_path = "../output/acdc/acdc-ef-lv.csv"
all_folders = sorted(glob.glob("../data/acdc/training/patient*"))
ef_list = []

for folder in all_folders:
    curr_patient = os.path.basename(folder)
    frame1, frame2, spacing = load_frames(folder)
    data = np.array([frame1, frame2])
    data[data != 3] = 0
    data[data == 3] = 1
    imp = ImplicitFunction(data, grid_spacing=0.25, spacing=tuple(spacing))  # spacing is (slice, y, x)

    interpt = np.array([imp.interpolate(0), imp.interpolate(1)])
    np.save(f"../output/acdc/{curr_patient}.npy", interpt)

    ef_convex = metrics.ejection_fraction_polygon(data)
    ef_surface = metrics.ejection_fraction_surface(imp)
    ef_interp = metrics.ejection_fraction(interpt)
    ef_gt = metrics.ejection_fraction(data)
    print(f"{curr_patient}: polygons {ef_convex:.2f}, interpolation {ef_interp:.2f}, surface {ef_surface:.2f}, ground truth {ef_gt:.2f}")

    ef_list.append({'patient': curr_patient,
                    'convex': ef_convex, 'interp': ef_interp, 'surface': ef_surface, 'gt': ef_gt,
                    'convex_diff': ef_convex - ef_gt, 'interp_diff': ef_interp - ef_gt,
                    'surface_diff': ef_surface - ef_gt})

with open(output_csv_path, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=list(ef_list[0]))
    writer.writeheader()
    writer.writerows(ef_list)

## Ejection fraction error plot

In [ ]:
with open(output_csv_path, newline='') as csvfile:
    # Paper Fig. 10 plots the error of the EF computed from the implicit function (interp_diff).
    diff_list = [float(row['interp_diff']) for row in csv.DictReader(csvfile)]

abs_diff_list = [abs(ef) for ef in diff_list]
print('Average:', np.mean(abs_diff_list))
print('Median:', np.median(abs_diff_list))
print('Std:', np.std(abs_diff_list))
print('Min:', np.min(diff_list))
print('Max:', np.max(diff_list))

patients = range(1, len(abs_diff_list) + 1)
plt.title('Ejection fraction absolute estimation error')
plt.xlabel('Patient ID')
plt.ylabel('Ground truth difference')
plt.plot(patients, abs_diff_list)
plt.scatter(patients, abs_diff_list, color='red', s=20)
plt.show()

## Segmentation metrics of the reconstructed frames

In [ ]:
scores = {'dice': [], 'iou': [], 'precision': [], 'recall': []}
for folder in all_folders:
    interpt = np.load(f"../output/acdc/{os.path.basename(folder)}.npy")
    frame1, frame2, _ = load_frames(folder)
    for pred, gt in [(interpt[0], frame1), (interpt[1], frame2)]:
        pred, gt = pred != 0, gt == 3  # the fit used the LV label only
        scores['dice'].append(metrics.dice(pred, gt))
        scores['iou'].append(metrics.jaccard(pred, gt))
        scores['precision'].append(metrics.precision(pred, gt))
        scores['recall'].append(metrics.recall(pred, gt))

for name, values in scores.items():
    print(f'Average {name}:', np.mean(values))